# Business Insights for Power BI Dashboard
This notebook prepares the cleaned dataset and summary tables needed for the Power BI dashboard. It creates the dashboard-ready fields and saves exports that support KPI cards, risk analysis, and financial behavior visuals.

In [7]:
#-----------------------------------------
# 1. Setup and Imports
#-----------------------------------------

import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name in ["Notebooks", "notebooks"] else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", os.getcwd())

Current working directory: C:\Users\venut\OneDrive\Documents\Projects\credit_risk_project


In [8]:
#-----------------------------------------
# 2. Load Raw Dataset
#-----------------------------------------

DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
# CORRECT — load only needed columns in chunks
NEEDED_COLS = [
    "loan_status", "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "purpose", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "dti", "delinq_2yrs",
    "fico_range_low", "fico_range_high", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc", "application_type"
]

chunks = []
for chunk in pd.read_csv(
    DATASET_PATH,
    usecols=NEEDED_COLS,
    chunksize=100_000,
    low_memory=False
):
    chunks.append(chunk)
df_raw = pd.concat(chunks, ignore_index=True)

# Clean interest rate
if df_raw["int_rate"].dtype == object:
    df_raw["int_rate"] = df_raw["int_rate"].astype(str).str.replace("%", "", regex=False)
    df_raw["int_rate"] = pd.to_numeric(df_raw["int_rate"], errors="coerce")

print("Loaded rows:", len(df_raw))



Loaded rows: 2260701


In [9]:
#-----------------------------------------
# 3. Create Dashboard Fields
#-----------------------------------------

df = df_raw.copy()

# is_default: only meaningful for resolved loans — marked NaN for unresolved
resolved_mask = df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])
df["is_default"] = np.where(
    resolved_mask,
    df["loan_status"].isin(["Charged Off", "Default"]).astype(int),
    np.nan
)

# Average FICO
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

# Numeric term
df["loan_term_numeric"] = pd.to_numeric(
    df["term"].str.extract(r"(\d+)")[0], errors="coerce"
)

# Loan-to-income ratio
df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)

# Income bracket
income_bins = [0, 30000, 60000, 90000, 120000, 150000, np.inf]
income_labels = ["0-30K", "30K-60K", "60K-90K", "90K-120K", "120K-150K", ">150K"]
df["income_bracket"] = pd.cut(df["annual_inc"], bins=income_bins, labels=income_labels)

# Default category label (only for resolved loans)
df["default_rate_category"] = np.where(
    resolved_mask,
    np.where(df["loan_status"].isin(["Charged Off", "Default"]), "Bad Loan", "Good Loan"),
    "Unresolved"
)

print("Dashboard fields created.")
print(df[["loan_status", "is_default", "default_rate_category"]].value_counts().head(10))

Dashboard fields created.
loan_status  is_default  default_rate_category
Fully Paid   0.0         Good Loan                1076751
Charged Off  1.0         Bad Loan                  268559
Default      1.0         Bad Loan                      40
Name: count, dtype: int64


In [10]:
#-----------------------------------------
# 4. Basic Validation Checks
#-----------------------------------------

# is_default should only be 0, 1, or NaN — never other values
valid_defaults = df["is_default"].dropna().isin([0, 1]).all()
assert valid_defaults, "is_default contains unexpected values"

# FICO average sanity check
assert df["fico_avg"].dropna().between(300, 850).all(), "fico_avg out of expected range"

# Loan amount must be positive
assert df["loan_amnt"].dropna().gt(0).all(), "loan_amnt contains non-positive values"

print("All validation checks passed.")

All validation checks passed.


In [11]:
#-----------------------------------------
# 5. Save Dashboard Dataset
#-----------------------------------------

selected_cols = [
    "loan_amnt", "term", "loan_term_numeric", "int_rate", "installment",
    "grade", "sub_grade", "purpose", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "dti",
    "delinq_2yrs", "fico_range_low", "fico_range_high", "fico_avg",
    "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "application_type", "loan_status",
    "is_default", "default_rate_category",
    "income_bracket", "loan_to_income"
]

# Keep only columns that exist in the dataframe
selected_cols = [c for c in selected_cols if c in df.columns]

CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "dashboard_data.csv"
df[selected_cols].to_csv(CLEAN_PATH, index=False)
print("Saved dashboard dataset to:", CLEAN_PATH)
print("Columns exported:", len(selected_cols))
print("Rows exported:", len(df))

Saved dashboard dataset to: C:\Users\venut\OneDrive\Documents\Projects\credit_risk_project\data\processed\dashboard_data.csv
Columns exported: 28
Rows exported: 2260701


In [12]:
#-----------------------------------------
# 6. KPI Summary for Dashboard
#-----------------------------------------

df_resolved = df[resolved_mask].copy()

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Loans (All)",
        "Resolved Loans",
        "Default Rate (Resolved Loans Only)",
        "Average Loan Amount",
        "Average Interest Rate"
    ],
    "Value": [
        len(df),
        len(df_resolved),
        df_resolved["is_default"].mean() * 100,
        df["loan_amnt"].mean(),
        df["int_rate"].mean()
    ]
})
print(kpi_summary.to_string(index=False))

                            Metric        Value
                 Total Loans (All) 2.260701e+06
                    Resolved Loans 1.345350e+06
Default Rate (Resolved Loans Only) 1.996499e+01
               Average Loan Amount 1.504693e+04
             Average Interest Rate 1.309283e+01


In [13]:
#-----------------------------------------
# 7. Risk Analysis by Key Dimensions
#-----------------------------------------
# All risk tables use resolved loans only

risk_by_grade = (
    df_resolved.groupby("grade", observed=True)["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
)

risk_by_term = (
    df_resolved.groupby("term", observed=True)["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
)

risk_by_purpose = (
    df_resolved.groupby("purpose", observed=True)["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
    .sort_values("default_rate", ascending=False)
)

risk_by_income = (
    df_resolved.groupby("income_bracket", observed=True)["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
)

# Save all
DATASET_DIR = PROJECT_ROOT / "data" / "processed"
risk_by_grade.to_csv(DATASET_DIR / "dashboard_risk_by_grade.csv", index=False)
risk_by_term.to_csv(DATASET_DIR / "dashboard_risk_by_term.csv", index=False)
risk_by_purpose.to_csv(DATASET_DIR / "dashboard_risk_by_purpose.csv", index=False)
risk_by_income.to_csv(DATASET_DIR / "dashboard_risk_by_income.csv", index=False)

print("Saved aggregate dashboard tables.")
print("\nDefault Rate by Grade:")
print(risk_by_grade)
print("\nDefault Rate by Term:")
print(risk_by_term)

Saved aggregate dashboard tables.

Default Rate by Grade:
  grade  default_rate
0     A      0.060427
1     B      0.133867
2     C      0.224413
3     D      0.303867
4     E      0.384823
5     F      0.452042
6     G      0.499343

Default Rate by Term:
         term  default_rate
0   36 months      0.159955
1   60 months      0.324485


In [14]:
#---------------------------------------------
# 8. Display Sample Data and Summary Statistics
#---------------------------------------------

print(df[["loan_status", "is_default", "grade",
          "loan_term_numeric", "fico_avg", "income_bracket"]].head(10))

print("\nResolved loan default rate:",
      f"{df_resolved['is_default'].mean():.4f}")
print("(This excludes Current, Late, Grace Period loans — more accurate than using all rows)")

  loan_status  is_default grade  loan_term_numeric  fico_avg income_bracket
0  Fully Paid         0.0     C               36.0     677.0        30K-60K
1  Fully Paid         0.0     C               36.0     717.0        60K-90K
2  Fully Paid         0.0     B               60.0     697.0        60K-90K
3     Current         NaN     C               60.0     787.0       90K-120K
4  Fully Paid         0.0     F               60.0     697.0       90K-120K
5  Fully Paid         0.0     C               36.0     692.0        30K-60K
6  Fully Paid         0.0     B               36.0     682.0          >150K
7  Fully Paid         0.0     B               36.0     707.0        60K-90K
8  Fully Paid         0.0     A               36.0     687.0        60K-90K
9  Fully Paid         0.0     B               36.0     702.0        30K-60K

Resolved loan default rate: 0.1996
(This excludes Current, Late, Grace Period loans — more accurate than using all rows)
